## MDCKII label quantification

In [ ]:
1+1

In [ ]:
import tifffile as tiff
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.ndimage import binary_dilation, gaussian_filter
from scipy.ndimage import label as nd_label
from skimage.measure import regionprops, marching_cubes, mesh_surface_area
from skimage.morphology import disk
import inspect

In [ ]:
# Define the root file name (without extension or suffixes)
date= "26.02.10"
file_name_root = "mask_intersect_1"

In [ ]:
# Labels for feature extraction
file_name_1 = f"{file_name_root}_preprocessed.tif"

labels = tiff.imread(f"./../{date}_MDCKII/masks_intersect/{file_name_1}")  # shape: (Z, Y, X)

print(f"Shape:     {labels.shape}")
print(f"Nº cells: {len(np.unique(labels)) - 1}")
print(f"Dtype:     {labels.dtype}")
print(f"Memory:   {labels.nbytes / 1e6:.1f} MB")

In [ ]:
# Labels for final representation
file_name_2 = f"{file_name_root}_noborder.tif"

labels_noborder = tiff.imread(f"./../{date}_MDCKII/masks_intersect/{file_name_2}")  # shape: (Z, Y, X)

print(f"Shape:     {labels_noborder.shape}")
print(f"Nº cells: {len(np.unique(labels_noborder)) - 1}")
print(f"Dtype:     {labels_noborder.dtype}")
print(f"Memory:   {labels_noborder.nbytes / 1e6:.1f} MB")

## Cellular features


In [ ]:
res_z = 0.3  # µm/px in Z
res_xy = 0.6 # µm/px in XY

### Cell position on the Z axis

In [ ]:
# Z centroid of each label, relative to a global epithelial surface

# Determine the basal Z of the epithelium: the highest Z where the fraction of zero pixels is below a threshold
Z, Y, X = labels.shape
total_pixels = Y * X

zero_fraction = np.array([
    (labels[z, :, :] == 0).sum() / total_pixels
    for z in range(Z)])

umbral = 0.70  # 70% set as threshold for basal Z determination

z_base  = np.max(np.where(zero_fraction < umbral)[0])

print(f"Z basal  (base):  {z_base}")

# Visualize the fraction of zero pixels along Z and the basal Z determination
plt.plot(zero_fraction)
plt.axhline(umbral, color='r', linestyle='--', label=f'umbral {umbral*100:.0f}%')
plt.axvline(z_base,  color='b', linestyle='--', label=f'Z basal  = {z_base}')
plt.xlabel('Z')
plt.ylabel('Fracción de píxeles = 0')
plt.title('Base y techo epitelial')
plt.legend()
plt.show()

# Z centroid of the complete cell: mean of all Z coordinates of the object
all_z, all_y, all_x = np.where(labels > 0)
all_labels = labels[all_z, all_y, all_x]

# Calculate Z centroid by label vectorized with pandas
df_z = (pd.DataFrame({'label': all_labels, 'z': all_z})
    .groupby('label')['z']
    .mean()
    .reset_index()
    .rename(columns={'z': 'centroid_cell_z'}))

# Relativize the Z centroid with respect to the base
df_z['centroid_cell_z_norm'] = z_base - df_z['centroid_cell_z']

# Convert the normalized Z centroid to micrometers
df_z['centroid_cell_z_norm_um'] = df_z['centroid_cell_z_norm'] * res_z
df_z.drop(columns=['centroid_cell_z_norm'], inplace=True)

df_props = df_z
print(df_props.head(10)) #low values imply that the centroid is closer to the basal surface, while high values imply that the cell is closer to the apical surface.

### Identify cells from the borders

In [ ]:
labels_border = np.setdiff1d(np.unique(labels),np.unique(labels_noborder))

df_props['border'] = df_props['label'].isin(labels_border)
df_props[['label', 'border']].head(15)

### General morphology

In [ ]:
def major_surface_component(coords):
    """
    Given an array of ZYX coordinates of a surface, returns only
    the coordinates belonging to the largest connected component (in 2D, YX).
    """
    if len(coords) == 0:
        return coords

    ys = coords[:, 1].astype(int)
    xs = coords[:, 2].astype(int)
    y_off, x_off = ys.min(), xs.min()

    # Project the coordinates onto the YX plane and create a binary image
    img = np.zeros((ys.max() - y_off + 1, xs.max() - x_off + 1), dtype=np.uint8)
    img[ys - y_off, xs - x_off] = 1

    # Label connected components in the binary image
    labeled, n_components = nd_label(img)

    if n_components <= 1:
        return coords # the entire surface is already a single component

    # Keep only the coordinates corresponding to the largest connected component
    component_sizes = np.bincount(labeled.ravel())
    component_sizes[0] = 0  # ignorar fondo
    largest = component_sizes.argmax()

    # Create a mask for the largest component and filter the original coordinates
    mask_largest = labeled == largest
    keep = mask_largest[ys - y_off, xs - x_off].astype(bool)

    return coords[keep]

def calculate_perimeter(coords):
    """
    Given ZYX coordinates of a surface, returns the 2D perimeter
    as the sum of exposed exterior sides of the border pixels
    in the YX projection (4-connectivity).
    """
    if len(coords) == 0:
        return np.nan

    ys = coords[:, 1].astype(int)
    xs = coords[:, 2].astype(int) #keep only the YX coordinates for perimeter calculation

    y_off, x_off = ys.min(), xs.min()
    img = np.zeros((ys.max() - y_off + 3, xs.max() - x_off + 3), dtype=np.uint8)
    img[ys - y_off + 1, xs - x_off + 1] = 1 #binary crop from the 

    shifts = [(-1, 0), (1, 0), (0, -1), (0, 1)] #shifts for 4-connectivity (up, down, left, right)
    perimeter = 0
    
    for dy, dx in shifts: # iterate over the shifts to check for exposed sides
        shifted = np.roll(img, shift=(dy, dx), axis=(0, 1)) # shift each pixel of the image to check for neighbors
        # Set the border pixels to 0 after shifting to avoid wrap-around effects
        if dy == -1: shifted[-1, :] = 0
        if dy ==  1: shifted[0,  :] = 0
        if dx == -1: shifted[:, -1] = 0
        if dx ==  1: shifted[:,  0] = 0

        # Exposed pixels are those that are occupied in img and have a neighbor that is 0 in the shifted image
        exposed = img & (shifted == 0)
        perimeter += int(exposed.sum()) #adds the number of exposed pixel sides

    return perimeter

In [ ]:
# Calculate apical and basal areas, height, respective perimeters and respective shape indexes (regularity)
rows = []
coords_apical_dict = {}
coords_basal_dict  = {}
Z_max = labels.shape[0]

for label in np.unique(labels):
    print(label)
    if label == 0:
        continue

    if df_props.loc[df_props['label'] == label, 'border'].values[0] == True: #omit border cells
        rows.append({
        'label'     : label,
        'height_um'             : np.nan,
        'area_apical_um2'        : np.nan,
        'area_basal_um2'         : np.nan,
        'logratio_ap_bas'       : np.nan,
        'shape_index_apical' : np.nan,
        'shape_index_basal'  : np.nan,      
        })
        continue
    
    mask      = (labels == label)
    footprint = mask.any(axis=0)
    yx_coords = np.argwhere(footprint) # XY coordinates where the label exists in at least one Z

    apical_pixels = 0
    basal_pixels  = 0
    apical_coords = []
    basal_coords  = []

    # Apical and basal surfaces: for each (y, x) where the cell exists, identify apical and basal voxels
    # Lower Zs are apical, higher Zs are basal
    # Apical surface: lowest Z in the cell with a 0 in a lower Z is apical surface
    # Basal surface: highest Z in the cell with a 0 in a higher Z is basal surface

    for y, x in yx_coords:
        z_col      = mask[:, y, x] # Z for this XY
        z_occupied = np.where(z_col)[0] # indices Z where the cell is present in this XY coordinate
        z_min = z_occupied[0]
        z_max = z_occupied[-1]

        if z_min == 0 or labels[z_min - 1, y, x] == 0:
            apical_coords.append((z_min, y, x)) #save ZYX coordinates of each apical pixel
        if z_max == Z_max - 1 or labels[z_max + 1, y, x] == 0:
            basal_coords.append((z_max, y, x)) #save ZYX coordinates of each basal pixel

    apical_coords = np.array(apical_coords) if apical_coords else np.empty((0, 3), dtype=int) #convert to array for easier subsequent calculations, or empty array if no coordinates
    basal_coords  = np.array(basal_coords)  if basal_coords  else np.empty((0, 3), dtype=int) #convert to array for easier subsequent calculations, or empty array if no coordinates
    # Keep only the largest connected component of the apical surface. We keep the original coordinates for curvature calculations.
    apical_coords_filt = major_surface_component(apical_coords)
    basal_coords_filt  = major_surface_component(basal_coords)
    apical_pixels = len(apical_coords_filt) #count the number of pixels in the filtered apical surface
    basal_pixels  = len(basal_coords_filt) #count the number of pixels in the filtered basal surface

    # Calculate cell height: difference in Z between basal and apical XY centroids
    # Apical and basal centroids (YX) and their corresponding Z values
    if len(apical_coords_filt) > 0:
        centroid_apical_y = apical_coords_filt[:, 1].mean()
        centroid_apical_x = apical_coords_filt[:, 2].mean()
        # Search in the apical array for the point closest to the XY centroid
        cy, cx = round(centroid_apical_y), round(centroid_apical_x)
        match = apical_coords_filt[(apical_coords_filt[:, 1] == cy) & (apical_coords_filt[:, 2] == cx)]
        if len(match) > 0:
            centroid_apical_z = float(match[0, 0])
        else:
            # If the exact XY centroid is not in the surface (e.g., C-shaped), take the closest point
            dists = (apical_coords_filt[:, 1] - centroid_apical_y)**2 + (apical_coords_filt[:, 2] - centroid_apical_x)**2
            centroid_apical_z = float(apical_coords_filt[dists.argmin(), 0])
    else:
        centroid_apical_y = centroid_apical_x = centroid_apical_z = np.nan
    if len(basal_coords_filt) > 0:
        centroid_basal_y = basal_coords_filt[:, 1].mean()
        centroid_basal_x = basal_coords_filt[:, 2].mean()
        cy, cx = round(centroid_basal_y), round(centroid_basal_x)
        match = basal_coords_filt[(basal_coords_filt[:, 1] == cy) & (basal_coords_filt[:, 2] == cx)]
        if len(match) > 0:
            centroid_basal_z = float(match[0, 0])
        else:
            dists = (basal_coords_filt[:, 1] - centroid_basal_y)**2 + (basal_coords_filt[:, 2] - centroid_basal_x)**2
            centroid_basal_z = float(basal_coords_filt[dists.argmin(), 0])
    else:
        centroid_basal_y = centroid_basal_x = centroid_basal_z = np.nan
    # Height: Z difference between basal and apical centroids
    height = abs(centroid_basal_z - centroid_apical_z) if not (np.isnan(centroid_apical_z) or np.isnan(centroid_basal_z)) else np.nan

    # 2D perimeters: count exposed sides on the border of each surface
    perimeter2D_apical = calculate_perimeter(apical_coords)
    perimeter2D_basal  = calculate_perimeter(basal_coords)
    shape_index_apical = np.sqrt(apical_pixels) / (perimeter2D_apical + 1e-10) if perimeter2D_apical > 0 else np.nan
    shape_index_apical = shape_index_apical / (np.sqrt(np.pi)/(2*np.pi)) #normalize shape index to a circle's
    shape_index_basal  = np.sqrt(basal_pixels)  / (perimeter2D_basal  + 1e-10) if perimeter2D_basal  > 0 else np.nan
    shape_index_basal  = shape_index_basal  / (np.sqrt(np.pi)/(2*np.pi)) #normalize shape index to a circle's

    rows.append({
        'label'              : label,
        'height_um'             : height * res_z, # convert to microns
        'area_apical_um2'        : apical_pixels * res_xy**2 if apical_pixels > 0 else np.nan, # convert to microns
        'area_basal_um2'         : basal_pixels * res_xy**2 if basal_pixels > 0 else np.nan, # convert to microns
        'logratio_ap_bas'       : np.log2(apical_pixels / basal_pixels) if basal_pixels > 0 else np.nan,
        'shape_index_apical' : shape_index_apical if apical_pixels > 0 else np.nan,
        'shape_index_basal'  : shape_index_basal if basal_pixels > 0 else np.nan,      
    })

    coords_apical_dict[label] = apical_coords_filt #save apical coordinates in a dictionary with the label as key
    coords_basal_dict[label]  = basal_coords_filt #save basal coordinates in a dictionary with the label as key

df_cosas = pd.DataFrame(rows)
df_props =df_props.drop(columns=['height_um', 'area_apical_um2',
                                 'area_basal_um2', 'logratio_ap_bas',
                                 'shape_index_apical', 'shape_index_basal',], errors='ignore') #safe drop to avoid confusions

df_props = df_props.merge(df_cosas, on='label')
print(f"Células procesadas: {len(df_props)}")
print(df_props[['label', 'border', 'area_apical_um2', 'area_basal_um2', 'logratio_ap_bas',
                'shape_index_apical', 'shape_index_basal']].head(10))
print(df_props[['shape_index_apical', 'shape_index_basal']].describe())

In [ ]:
# Compute the difference in shape index and area between apical and basal surfaces, normalized by their mean
df_props['dif_shape_index'] = df_props['shape_index_apical'] - df_props['shape_index_basal']
df_props['dif_area'] = 2*(df_props['area_apical_um2'] - df_props['area_basal_um2'])/(df_props['area_apical_um2'] + df_props['area_basal_um2'])

# Compute the aspect ratio, defined as the height of the cell divided by the square root of the average area of the apical and basal surfaces. This gives a measure of how elongated or flattened the cell is in relation to its surface area.
df_props['aspect_ratio'] = df_props['height_um'] / np.sqrt((df_props['area_apical_um2']+df_props['area_basal_um2'])/2)

In [ ]:
# Evaluate outliers
print(sum(df_props['logratio_ap_bas']>np.log2(3.0)))
print(df_props[df_props['logratio_ap_bas']>np.log2(3.0)][['label', 'area_apical_um2','area_basal_um2','logratio_ap_bas']])

print(sum(df_props['logratio_ap_bas']<np.log2(0.33)))
print(df_props[df_props['logratio_ap_bas']<np.log2(0.33)][['label', 'area_apical_um2','area_basal_um2','logratio_ap_bas']])

# Histogram of ratio apical/basal
plt.figure(figsize=(6, 4))
plt.hist(df_props['logratio_ap_bas'], bins=30)
plt.xlabel('Log Ratio Apical/Basal')
plt.ylabel('Frequency')
plt.title('Distribution of Apical/Basal log ratio')
plt.show()

In [ ]:
# Calculate volumes, surfaces, and sphericity for each cell

# Resolution parameters
vox_vol = res_z * res_xy ** 2  # volume of a voxel in µm³

rows = []

for r in regionprops(labels):
    label = r.label
    #print(label)

    # if there is a nan in any column of the row, skip (could be cells with basal or apical area 0, or on the border)
    row = df_props.loc[df_props['label'] == label]
    if row.iloc[0].isna().any() or row.iloc[0]['border']:
        rows.append({
        'label': label,
        'volume_um3': np.nan,
        'surface_um2': np.nan,
        'sphericity': np.nan
        })
        continue
    
    minz, miny, minx, maxz, maxy, maxx = r.bbox
    mask_crop = (labels[minz:maxz, miny:maxy, minx:maxx] == label) # binary mask, limited to the analysis area of the label's bounding box

    verts, faces, _, _ = marching_cubes(mask_crop,  # triangular mesh extraction from the cell
                                        level=0.5, # level of adjustment of the mesh to the mask) 
                                        spacing=(res_z, res_xy, res_xy)) # anisotropy of pixels in Z, Y, X (resolution for each dimension)
    
    surface = mesh_surface_area(verts, faces) # calculates the surface area from the vertices and faces, in µm²
    
    # Calculate volume from the triangular mesh (tetrahedron method). Coherent methods.
    v0 = verts[faces[:, 0]]
    v1 = verts[faces[:, 1]]
    v2 = verts[faces[:, 2]]
    cross = np.cross(v1, v2)
    signed_volumes = np.einsum('ij,ij->i', v0, cross)
    volume = np.abs(signed_volumes.sum()) / 6.0

    # Sphericity: 1 = perfect sphere, <1 = less spherical
    sphericity = (np.pi ** (1/3) * (6 * volume) ** (2/3)) / surface # formula of sphericity, compares the cell with a sphere of the same volume

    rows.append({
        'label'     : label,
        'volume_um3'    : volume,
        'surface_um2'   : surface,
        'sphericity': sphericity,
    })

df_sphericity = pd.DataFrame(rows)
df_props = df_props.drop(columns=['volume_um3', 'surface_um2', 'sphericity'], errors='ignore')
df_props = df_props.merge(df_sphericity, on='label')
print(df_props[['label', 'logratio_ap_bas', 'volume_um3', 'surface_um2', 'sphericity']].head(10))
print(df_props['sphericity'].describe().round(3))

In [ ]:
print(sum(df_props['sphericity']>1.0))
print(df_props[df_props['sphericity']>1.0][['label', 'sphericity']])

# Histogram of sphericity
plt.figure(figsize=(6, 4))
plt.hist(df_props['sphericity'], bins=30)
plt.xlabel('Sphericity')
plt.ylabel('Frequency')
plt.title('Distribution of Sphericity')
plt.show()

### Elongation and Z angle

In [ ]:
# # Elongation: ratio between the major and minor axes of the cell, calculated from its 3D mask, corrected for voxel anisotropy (physical units, µm)
# # Angle Z: inclination angle of the elongation axis with respect to the XY plane (0° = horizontal, 90° = vertical), calculated in physical units (µm), corrected for voxel anisotropy
# # A cell with elongation ~1, the inclination is not relevant, but in elongated cells it can indicate vertical (columnar) vs horizontal (flat) orientation

# voxel_size = (res_z, res_xy, res_xy)  # (Z, Y, X) in µm/px — applied to both elongation and angle_z

# props    = regionprops(labels)
# rows_elo = []

# for r in props:
#     label = r.label

#     row = df_props.loc[df_props['label'] == label]
#     if row.iloc[0].isna().any() or row.iloc[0]['border']:
#         rows_elo.append({'label': label, 'elongation': np.nan, 'angle_z': np.nan})
#         continue

#     # ── Inertia tensor in physical units (µm), corrected for anisotropy ──
#     coords    = r.coords.astype(float)              # (N, 3) ZYX in voxels
#     coords_um = coords * np.array(voxel_size)        # (N, 3) ZYX in µm
#     coords_centered = coords_um - coords_um.mean(axis=0)
#     inertia_um = coords_centered.T @ coords_centered / len(coords_centered)

#     eigvals_um, eigvecs_um = np.linalg.eigh(inertia_um)
#     order   = np.argsort(eigvals_um)[::-1]  # descendente
#     eigvals_um = eigvals_um[order]
#     eigvecs_um = eigvecs_um[:, order]

#     # Elongation: ratio between the major and minor axes in µm, adjusted to avoid division by zero
#     elongation = np.sqrt(eigvals_um[0]) / np.sqrt(eigvals_um[-1]) if eigvals_um[-1] > 0 else np.nan

#     # NOTE: in physical (µm) space, anisotropic scaling can flip the relative magnitude of eigenvalues vs voxel space. Empirically verified: in this
#     # dataset, the axis matching the validated biological orientation corresponds to the LARGEST eigenvalue in µm space (opposite of voxel space).
#     eje_elongacion = eigvecs_um[:, 0]  # greatest eigenvalue = axis of greatest elongation
#     # Angle Z: inclination of the elongation axis with respect to the XY plane, in physical units
#     angle_z = np.degrees(np.arcsin(np.clip(abs(eje_elongacion[0]), 0, 1)))

#     rows_elo.append({
#         'label'      : label,
#         'elongation' : elongation,
#         'angle_z'    : angle_z,
#     })

# df_elo   = pd.DataFrame(rows_elo)
# df_props = df_props.drop(columns=['elongation', 'angle_z'], errors='ignore')
# df_props = df_props.merge(df_elo, on='label')

# print(df_props[['label', 'elongation', 'angle_z']].head(10))
# print(df_props[['elongation', 'angle_z']].describe().round(3))

In [ ]:
# Elongation: ratio between the major and minor axes of the cell, calculated from its 3D mask, corrected for voxel anisotropy (physical units, µm)
# Angle Z: inclination angle of the elongation axis with respect to the XY plane (0° = horizontal, 90° = vertical), calculated in voxel space (not corrected for anisotropy, as it better separates cell populations in this dataset)
# A cell with elongation ~1, the inclination is not relevant, but in elongated cells it can indicate vertical (columnar) vs horizontal (flat) orientation

# NOTE: in the inertia tensor, the axis of GREATEST geometric elongation corresponds to the SMALLEST eigenvalue
# (same as a rigid solid: a pencil's long axis is the one with the lowest moment of inertia). Verified empirically against bbox dimensions.

voxel_size = (res_z, res_xy, res_xy)  # (Z, Y, X) in µm/px — used only for elongation, not for angle_z

props    = regionprops(labels)
rows_elo = []

for r in props:
    label = r.label

    row = df_props.loc[df_props['label'] == label]
    if row.iloc[0].isna().any() or row.iloc[0]['border']:
        rows_elo.append({'label': label, 'elongation': np.nan, 'angle_z': np.nan})
        continue

    # ── Angle Z: calculated in voxel space (uncorrected), as validated visually ──
    inertia_voxel = r.inertia_tensor
    eigvals_v, eigvecs_v = np.linalg.eigh(inertia_voxel)
    order_v   = np.argsort(eigvals_v)[::-1]
    eigvecs_v = eigvecs_v[:, order_v]

    eje_elongacion = eigvecs_v[:, -1]  # smallest eigenvalue = axis of greatest elongation
    angle_z = np.degrees(np.arcsin(np.clip(abs(eje_elongacion[0]), 0, 1)))

    # ── Elongation: calculated in physical units (µm), corrected for anisotropy ──
    coords    = r.coords.astype(float)              # (N, 3) ZYX in voxels
    coords_um = coords * np.array(voxel_size)        # (N, 3) ZYX in µm
    coords_centered = coords_um - coords_um.mean(axis=0)
    inertia_um = coords_centered.T @ coords_centered / len(coords_centered)

    eigvals_um, _ = np.linalg.eigh(inertia_um)
    eigvals_um    = np.sort(eigvals_um)[::-1]  # descendente

    # Elongation: ratio between the major and minor axes in µm, adjusted to avoid division by zero
    # Note: smallest eigenvalue (eigvals_um[-1]) = axis of greatest elongation,
    # so the ratio uses the largest eigenvalue (width) over the smallest (length) for elongation > 1
    elongation = np.sqrt(eigvals_um[0]) / np.sqrt(eigvals_um[-1]) if eigvals_um[-1] > 0 else np.nan

    rows_elo.append({
        'label'      : label,
        'elongation' : elongation,
        'angle_z'    : angle_z,
    })

df_elo   = pd.DataFrame(rows_elo)
df_props = df_props.drop(columns=['elongation', 'angle_z'], errors='ignore')
df_props = df_props.merge(df_elo, on='label')

print(df_props[['label', 'elongation', 'angle_z']].head(10))
print(df_props[['elongation', 'angle_z']].describe().round(3))

### Curvatures

In [ ]:
def calculate_curvature(labels_vol, sigma, res_z, res_xy, superficie='apical', plot=True):
    """
    Calculate the average curvature on the apical or basal surface.
    labels_vol : 3D array of labels (Z, Y, X)
    sigma       : scale of smoothing in pixels. Allows to calculate curvature at different scales. A larger sigma smooths the surface more (tissue), while a smaller sigma preserves more local details (cellular).
    res_z       : resolution in Z (µm/px)
    res_xy      : resolution in XY (µm/px)
    superficie  : 'apical' or 'basal'
    plot        : show figure

    curvature   : curvature map in µm⁻¹
    z_global    : Z map of the surface in pixels
    nan_mask    : mask of pixels without tissue
    """
    tissue_mask = labels_vol > 0
    no_tissue   = ~tissue_mask.any(axis=0)

    if superficie == 'apical':
        z_global = np.argmax(tissue_mask, axis=0).astype(float)
    elif superficie == 'basal':
        flipped  = np.flip(tissue_mask, axis=0)
        z_global = (labels_vol.shape[0] - 1 - np.argmax(flipped, axis=0)).astype(float)
    else:
        raise ValueError("surface must be 'apical' or 'basal'")

    z_global[no_tissue] = np.nan
    nan_mask = np.isnan(z_global)

    # Smooth the surface using a Gaussian filter, filling NaNs with the median of the surface to avoid artifacts
    z_filled         = z_global.copy()
    z_filled[nan_mask] = np.nanmedian(z_filled)
    z_smooth         = gaussian_filter(z_filled, sigma=sigma)
    z_smooth_um      = z_smooth * res_z
    z_smooth_um[nan_mask] = np.nan

    # Calculte the curvature as the second derivative of the surface. The curvature is calculated using the formula for the mean curvature of a surface defined by z = f(x, y):
    dz_dy, dz_dx     = np.gradient(z_smooth_um, res_xy, res_xy) #first derivatives: ∂z/∂x and ∂z/∂y

    d2z_dx2   = np.gradient(dz_dx, res_xy, axis=1)   # ∂²z/∂x²
    d2z_dy2   = np.gradient(dz_dy, res_xy, axis=0)   # ∂²z/∂y²
    d2z_dxdy  = np.gradient(dz_dx, res_xy, axis=0)   # ∂²z/∂x∂y  (= ∂²z/∂y∂x)

    p, q  = dz_dx, dz_dy
    r, s, t = d2z_dx2, d2z_dxdy, d2z_dy2

    denom     = 2 * (1 + p**2 + q**2)**(3/2)
    curvature = ((1 + q**2)*r - 2*p*q*s + (1 + p**2)*t) / (denom + 1e-10) # curvature formula for a surface defined by z = f(x, y)
    curvature[nan_mask] = np.nan

    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(20, 6))
        im0 = axes[0].imshow(z_global,    cmap='inferno')
        plt.colorbar(im0, ax=axes[0], label=f'Z {superficie} (px)')
        axes[0].set_title(f'Global surface height {superficie}')
        axes[0].axis('off')

        im1 = axes[1].imshow(z_smooth,    cmap='inferno')
        plt.colorbar(im1, ax=axes[1], label=f'Z {superficie} smoothed (px)')
        axes[1].set_title(f'Global surface height smoothed (sigma={sigma} px = {sigma*res_xy:.1f} µm)')
        axes[1].axis('off')

        im2 = axes[2].imshow(curvature,   cmap='RdBu_r', vmin=-0.005, vmax=0.005)
        plt.colorbar(im2, ax=axes[2], label='Average curvature (µm⁻¹)')
        axes[2].set_title('Average curvature')
        axes[2].axis('off')

        plt.suptitle(f'Average curvature — {superficie}')
        plt.tight_layout()
        plt.show()

    return curvature, z_global, nan_mask


# Calculate the average curvature of the apical and basal surfaces at tissue and cellular scales
curvature_apical_tisular, _, _ = calculate_curvature(
    labels, sigma=23, res_z=res_z, res_xy=res_xy, superficie='apical') #returns whether the cell is in a tissue-scale apical vault or apical crypt

curvature_apical_celular, _, _ = calculate_curvature(
    labels, sigma=5, res_z=res_z, res_xy=res_xy, superficie='apical') #returns whether the cell has the shape of an apical peak (vault) or an apical basin (crypt)

curvature_basal_tisular, _, _ = calculate_curvature(
    labels, sigma=23, res_z=res_z, res_xy=res_xy, superficie='basal') #returns whether the cell is in an basal vault or a basal crypt

curvature_basal_celular, _, _ = calculate_curvature(
    labels, sigma=5, res_z=res_z, res_xy=res_xy, superficie='basal') #returns whether the cell has the shape of a basal peak (vault) or a basal basin (crypt)

In [ ]:
def asign_curvature(df_props, coords_dict, curvature_map):
    """
    Asign average curvature from a curvature map to each label, based on its ZYX surface coordinates.

    df_props      : dataframe with a 'label' column
    coords_dict   : label + array (N, 3) of ZYX coordinates of the surface of each label
    curvature_map : (Y, X) curvature map in µm⁻¹
    """
    # Determine the name of the curvature map variable in the calling frame to create appropriate column names
    frame     = inspect.currentframe().f_back
    map_name  = [k for k, v in frame.f_locals.items() if v is curvature_map]
    map_name  = map_name[0] if map_name else 'curvature'
    col_mean  = f'{map_name}_mean'
    col_std   = f'{map_name}_std'

    rows_curv = []

    for _, row in df_props.iterrows():
        label  = row['label']
        coords = coords_dict.get(label, np.empty((0, 3), dtype=int))

        if len(coords) == 0 or df_props.loc[df_props['label'] == label, 'border'].values[0] == True: #omit border or nan cells
            rows_curv.append({'label': label, col_mean: np.nan})
            continue

        ys = coords[:, 1] # Y coordinates of the surface of each label
        xs = coords[:, 2] # X coordinates of the surface of each label

        valid     = ((ys >= 0) & (ys < curvature_map.shape[0]) &
                     (xs >= 0) & (xs < curvature_map.shape[1]))
        curv_vals = curvature_map[ys[valid], xs[valid]] # values of curvature at the coordinates of the surface of each label
        curv_vals = curv_vals[~np.isnan(curv_vals)]

        rows_curv.append({
            'label'  : label,
            col_mean : float(np.mean(curv_vals)) if len(curv_vals) > 0 else np.nan,
            #col_std  : float(np.std(curv_vals))  if len(curv_vals) > 0 else np.nan, #the results are not very informative, so we can omit them for now
        })

    df_curv  = pd.DataFrame(rows_curv)
    df_props = df_props.drop(columns=[col_mean, col_std], errors='ignore')
    df_props = df_props.merge(df_curv, on='label', how='left')

    print(f"Added columns: {col_mean}")

    return df_props


# Asign average tissue and cellular curvature to each label, for both apical and basal surfaces
# Tisular curvatures values are expected to be lower (smoother curves)
df_props = asign_curvature(df_props, coords_apical_dict, curvature_apical_tisular)
df_props = asign_curvature(df_props, coords_apical_dict, curvature_apical_celular)
curvature_basal_tisular_inv = -curvature_basal_tisular #invert the sign so that basal vaults have positive curvature, same as apical vaults
df_props = asign_curvature(df_props, coords_basal_dict, curvature_basal_tisular_inv)
curvature_basal_celular_inv = -curvature_basal_celular #invert the sign so that basal vaults have positive curvature, same as apical vaults
df_props = asign_curvature(df_props, coords_basal_dict,  curvature_basal_celular_inv)

df_props[['curvature_apical_tisular_mean',
          'curvature_apical_celular_mean',
          'curvature_basal_tisular_inv_mean',
          'curvature_basal_celular_inv_mean']].head(20)

H=0.01 μm−1; radius of 100 μm
H=0.001 μm−1; radius of ∼1000 μm,
H=0.1 μm−1; radius of ∼10 μm.

### Neighbors

In [ ]:
def safe_set(val):
    """ set ignoring NaN y None."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return set()
    return {v for v in val if v == v}  # v == v is False for NaN

#### Apical and basal neighbors based on 2-D apical and basal surfaces

In [ ]:
# Precalculate the apical and basal 2D maps for visualization
tissue_mask         = labels > 0
no_tissue           = ~tissue_mask.any(axis=0)
Y, X                = labels.shape[1], labels.shape[2]
ys_g, xs_g          = np.meshgrid(np.arange(Y), np.arange(X), indexing='ij')

apical_z_map        = np.argmax(tissue_mask, axis=0)
apical_map          = labels[apical_z_map, ys_g, xs_g].astype(int)
apical_map[no_tissue] = 0

tissue_flipped      = np.flip(tissue_mask, axis=0)
basal_z_map         = labels.shape[0] - 1 - np.argmax(tissue_flipped, axis=0)
basal_map           = labels[basal_z_map, ys_g, xs_g].astype(int)
basal_map[no_tissue] = 0

# Visualize the apical and basal maps, with a random color assigned by their label value
n_labels = int(labels.max())
rng = np.random.default_rng(seed=69)  # random seed
random_colors = rng.random((n_labels, 3))  # random RGB colors for each label
colors = np.vstack([[0, 0, 0], random_colors])  # 0 = black
cmap = mcolors.ListedColormap(colors)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(apical_map, cmap=cmap, vmin=0, vmax=n_labels)
axes[0].set_title('Mapa Apical')
axes[0].axis('off')
axes[1].imshow(basal_map, cmap=cmap, vmin=0, vmax=n_labels)
axes[1].set_title('Mapa Basal')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate apical and basal neighbors with a minimum contact threshold in pixels, using the 2D maps as reference

struct    = disk(2) # structure of dilation: disk of radius 2 (dilates 2 pixels in YX)
umbral_px = 6.0 # number of pixels in the dilated ring to consider a neighbor
rows_nb   = []

for label in df_props['label']:
    print(f'Label: {label}')

    if df_props.loc[df_props['label'] == label].iloc[0].isna().any(): # if there is a nan in any column of the row, skip (could be cells with basal or apical area 0, or on the border)
        rows_nb.append({
        'label'                  : label,
        'apical_neighbors_n'     : np.nan,
        'apical_neighbors_labels': np.nan,
        'basal_neighbors_n'      : np.nan,
        'basal_neighbors_labels' : np.nan,
        'dif_neighbors'          : np.nan,
        'potential_scutoid'      : np.nan,
        })
        continue

    # Apical
    mask_ap   = (apical_map == label)
    dil_ap    = binary_dilation(mask_ap, structure=struct)
    ring_ap   = dil_ap & ~mask_ap
    vals_ap   = apical_map[ring_ap] # label values in the dilated apical ring
    vals_ap   = vals_ap[(vals_ap != 0) & (vals_ap != label)]
    u_ap, c_ap = np.unique(vals_ap, return_counts=True) # u_ap are the labels of the apical neighbors, c_ap are the pixel contact counts
    ap_neighbors = {int(v) for v, c in zip(u_ap, c_ap) if c >= umbral_px} # apical neighbors that meet the minimum contact threshold

    # Basal
    mask_ba   = (basal_map == label)
    dil_ba    = binary_dilation(mask_ba, structure=struct)
    ring_ba   = dil_ba & ~mask_ba
    vals_ba   = basal_map[ring_ba] # label values in the dilated basal ring
    vals_ba   = vals_ba[(vals_ba != 0) & (vals_ba != label)]
    # print(f'Pixels in the dilated basal ring of cell {label}: {np.sum(ring_ba)}')
    u_ba, c_ba = np.unique(vals_ba, return_counts=True) # u_ba are the labels of the basal neighbors, c_ba are the pixel contact counts
    ba_neighbors = {int(v) for v, c in zip(u_ba, c_ba) if c >= umbral_px} # basal neighbors that meet the minimum contact threshold

    # Scutoids
    union_neighbors     = ap_neighbors | ba_neighbors
    shared_neighbors    = ap_neighbors & ba_neighbors
    dif_vecinos         = len(union_neighbors) - len(shared_neighbors)
    potencial_escutoide = dif_vecinos > 0

    rows_nb.append({
        'label'                  : label,
        'apical_neighbors_n'     : len(ap_neighbors),
        'apical_neighbors_labels': list(ap_neighbors),
        'basal_neighbors_n'      : len(ba_neighbors),
        'basal_neighbors_labels' : list(ba_neighbors),
        'dif_neighbors'          : dif_vecinos,
        'potential_scutoid'      : potencial_escutoide,
    })

df_nb    = pd.DataFrame(rows_nb)
df_props = df_props.drop(columns=['apical_neighbors_n', 'apical_neighbors_labels',
                                   'basal_neighbors_n',  'basal_neighbors_labels',
                                   'dif_neighbors', 'potential_scutoid'], errors='ignore')
df_props = df_props.merge(df_nb, on='label')

print(df_props[['label', 'apical_neighbors_n', 'basal_neighbors_n',
                'dif_neighbors', 'potential_scutoid']].head(10))
print(f"Escutoides: {df_props['potential_scutoid'].sum()} ({df_props['potential_scutoid'].mean()*100:.1f}%)")

In [ ]:
#histogram of apical and basal neighbors
print(df_props[['apical_neighbors_n', 'basal_neighbors_n']].describe().round(3))

plt.figure(figsize=(8, 5))
plt.hist(df_props['apical_neighbors_n'].dropna(), bins=100)
plt.xlabel('Number of apical neighbors')
plt.ylabel('Number of cells')
plt.title('Distribution of number of apical neighbors per cell')
plt.show()

# show the 10 labels with the most/least apical neighbors
#top_vecinos = df_props.sort_values('apical_neighbors_n', ascending=False).head(10)
top_vecinos = df_props.sort_values('apical_neighbors_n', ascending=True).head(10)
print(top_vecinos[['label', 'apical_neighbors_n', 'apical_neighbors_labels']])

plt.figure(figsize=(8, 5))
plt.hist(df_props['basal_neighbors_n'].dropna(), bins=100)
plt.xlabel('Number of basal neighbors')
plt.ylabel('Number of cells')
plt.title('Distribution of number of basal neighbors per cell')
plt.show()

# show the 10 labels with the most/least basal neighbors
#top_vecinos = df_props.sort_values('basal_neighbors_n', ascending=False).head(10)
top_vecinos = df_props.sort_values('basal_neighbors_n', ascending=True).head(10)
print(top_vecinos[['label', 'basal_neighbors_n', 'basal_neighbors_labels']])

In [ ]:
# Scutoid fast analysis
print(f"Total no border cells:{len(df_props[df_props['border'] == False])}")
print(f"Scutoids: {df_props['potential_scutoid'].sum()}")
print(f"Scutoid percentage: {df_props['potential_scutoid'].mean()*100:.1f}%")
# 
df_props.nlargest(40, 'dif_neighbors')[['label', 'area_apical_um2', 'area_basal_um2','logratio_ap_bas',
                                       'apical_neighbors_n', 'apical_neighbors_labels', 
                                       'basal_neighbors_n', 'basal_neighbors_labels', 'dif_neighbors']]

#### 3D neighbors, using apical and basal neighbors

In [ ]:
def get_neighbor_counts(coords, labels_vol, label, pad=2):
    """
    Generates a dict were all potential neighbors labels for cells in a disk of radius 'pad' are saved, along with the contact pixel counts
    The filtering is done externally.
    """
    if len(coords) == 0:
        return {}

    zs, ys, xs = coords[:, 0], coords[:, 1], coords[:, 2]

    z0 = max(zs.min() - pad, 0);  z1 = min(zs.max() + pad + 1, labels_vol.shape[0])
    y0 = max(ys.min() - pad, 0);  y1 = min(ys.max() + pad + 1, labels_vol.shape[1])
    x0 = max(xs.min() - pad, 0);  x1 = min(xs.max() + pad + 1, labels_vol.shape[2])

    local      = labels_vol[z0:z1, y0:y1, x0:x1]
    mask_local = np.zeros_like(local, dtype=bool)
    mask_local[zs - z0, ys - y0, xs - x0] = True

    struct_2d = disk(pad)
    struct_3d = struct_2d[np.newaxis, :, :]
    dilated   = binary_dilation(mask_local, structure=struct_3d)
    border    = dilated & ~mask_local

    vals = local[border]
    vals = vals[(vals != 0) & (vals != label)]

    unique, counts = np.unique(vals, return_counts=True)
    return {int(v): int(c) for v, c in zip(unique, counts)}

In [ ]:
# 3D neighbor calculation with semantic filtering
umbral_vx_semantico = 175  # strict threshold for non-apical/basal neighbors

props = regionprops(labels)
rows3D  = []

for r in props:
    label = r.label
    print(f"Procesando label {label}")

    row = df_props.loc[df_props['label'] == label]
    if row.iloc[0].isna().any() or row.iloc[0]['border']: # if there is a nan in any column of the row, skip (could be cells with basal or apical area 0, or on the border)
        rows3D.append({
        'label'                  : label,
        '3D_neighbors_n'         : np.nan,
        '3D_neighbors_labels'    : np.nan})
        continue

    coords = np.array(r.coords)  # ZYX

    # Apical and basal neighbors
    row_props = df_props.loc[df_props['label'] == label].iloc[0]
    vecinos_apicobasales = safe_set(row_props['apical_neighbors_labels']) | \
                           safe_set(row_props['basal_neighbors_labels'])
    # 3D neighbors
    vecinos_con_conteo = get_neighbor_counts(coords, labels, label, pad=2)
    # Semantic filtering: keep only apical/basal neighbors or those with enough contact pixels
    vecinos_finales = set()
    for vecino, n_px in vecinos_con_conteo.items():
        if vecino in vecinos_apicobasales:
            vecinos_finales.add(vecino)           # apical/basal
        elif n_px >= umbral_vx_semantico:
            vecinos_finales.add(vecino)           # only if the contact is high enough (>= 175 pixels)

    rows3D.append({
        'label'               : label,
        '3D_neighbors_n'      : len(vecinos_finales),
        '3D_neighbors_labels' : list(vecinos_finales),
    })

df_neighbors = pd.DataFrame(rows3D)
df_props = df_props.drop(columns=['3D_neighbors_n', '3D_neighbors_labels'], errors='ignore')
df_props = df_props.merge(df_neighbors, on='label')

print(df_props[['label', '3D_neighbors_n', '3D_neighbors_labels']].head(10))

In [ ]:
# 3d neighbors histogram
plt.figure(figsize=(8, 5))
plt.hist(df_props['3D_neighbors_n'].dropna(), bins=100)
plt.xlabel('Número de vecinos 3D')
plt.ylabel('Cantidad de células')
plt.title('Distribución de número de vecinos 3D por célula')
plt.show()

print(df_props['3D_neighbors_n'].describe())
# show the 10 labels with the most/least 3D neighbors
#top_vecinos = df_props.sort_values('3D_neighbors_n', ascending=False).head(50)
top_vecinos = df_props.sort_values('3D_neighbors_n', ascending=True).head(50)
print(top_vecinos[['label', '3D_neighbors_n', '3D_neighbors_labels']])

#### Neighbor verifications

In [ ]:
# Numerical inconsistency: the number of 3D neighbors should be greater than or equal to the number of apical or basal neighbors
inconsistencia_n = 0
for idx, row in df_props.iterrows():
    n3d     = row['3D_neighbors_n']     if not pd.isna(row['3D_neighbors_n'])     else 0
    n_apic  = row['apical_neighbors_n'] if not pd.isna(row['apical_neighbors_n']) else 0
    n_basal = row['basal_neighbors_n']  if not pd.isna(row['basal_neighbors_n'])  else 0

    if n3d < n_apic or n3d < n_basal:
        inconsistencia_n += 1
        print(f"Numeric inconsistency in {row['label']}! "
              f"3D neighbors: {n3d}, Apical: {n_apic}, Basal: {n_basal}")

print(f"Number of numerical inconsistencies: {inconsistencia_n}")

# Semantic inconsistency: the set of 3D neighbors should be equal to the union of apical and basal neighbors
labels_3D_problematicos = []
inconsistencia_s = 0
for idx, row in df_props.iterrows():
    s3d    = safe_set(row['3D_neighbors_labels'])
    s_apic = safe_set(row['apical_neighbors_labels'])
    s_bas  = safe_set(row['basal_neighbors_labels'])

    if s3d != (s_apic | s_bas):
        print(f"Semantic inconsistency in {row['label']}!")
        inconsistencia_s += 1
        vecinos_3D_unicos = s3d - (s_apic | s_bas)
        labels_3D_problematicos.append(vecinos_3D_unicos)
        print(f"3D neighbors unique (not apical or basal) in {row['label']}: {vecinos_3D_unicos}")

        vecinos_apicobasales_unicos = (s_apic | s_bas) - s3d
        if vecinos_apicobasales_unicos:
            print(f"Apical/Basal neighbors unique (not in 3D) in {row['label']}: {vecinos_apicobasales_unicos}")

print(f"Number of semantic inconsistencies: {inconsistencia_s}")

In [ ]:
# Add to the dataframe a column with the number of 3D neighbors that are neither apical nor basal, as an indicator of possible segmentation errors or cells with protrusions
def is_na_val(val):
    """Devuelve True si el valor es NaN escalar o None, False si es lista."""
    if val is None:
        return True
    if isinstance(val, float) and np.isnan(val):
        return True
    return False

def vecinos_3D_unicos(row):
    if is_na_val(row['3D_neighbors_labels']) or (is_na_val(row['apical_neighbors_labels']) and is_na_val(row['basal_neighbors_labels'])):
        return pd.Series({'inconsistency': np.nan })

    set_3d    = safe_set(row['3D_neighbors_labels'])
    set_apic  = safe_set(row['apical_neighbors_labels'])
    set_basal = safe_set(row['basal_neighbors_labels'])

    if len(set_3d) > len(set_apic | set_basal):
        unicos = set_3d - (set_apic | set_basal)
        return pd.Series({'inconsistency': 1, 'inconsistent_neighbors': list(unicos) })  # Inconsistency: 3D neighbors unique (not apical or basal)
    elif len(set_3d) < len(set_apic | set_basal):
        unicos = (set_apic | set_basal) - set_3d
        return pd.Series({'inconsistency': -1, 'inconsistent_neighbors': list(unicos) }) # Inconsistency: apical/basal neighbors unique (not in 3D)
    else:
        unicos= 0
        return pd.Series({'inconsistency': 0, 'inconsistent_neighbors': 0 })  # consistency: 3D neighbors coincide exactly with the union of apical and basal neighbors

df_props[['inconsistency','inconsistent_neighbors']] = df_props.apply(vecinos_3D_unicos, axis=1)
print(df_props[['label','inconsistency', 'inconsistent_neighbors']].head(20))

### Apical and basal local cell density

In [ ]:
# Calculate local density in a 2D projection (apical_map or basal_map) for each label

def density_2d(label, proj_map, y0, y1, x0, x1, circle,
                area_disco, area_disco_completo):

    crop = proj_map[y0:y1, x0:x1]

    vals = crop[circle]
    vals = vals[(vals != 0) & (vals != label)]
    n_labels = len(np.unique(vals))

    fraccion_area = area_disco / area_disco_completo # fraction of the area of the disk that is inside the image (useful for cells near the edge)
    local_density = (n_labels / area_disco if area_disco > 0 else np.nan) # number of cells per um² (in the disk of each label)

    return local_density, round(fraccion_area, 3)

In [ ]:
# Local density 2D: number of labels within a radius XY (px) relative to the centroid of each label using the apical and basal projections

radio_xy = 75
rows = []
centroids = {r.label: r.centroid for r in props}

for _, row in df_props.iterrows():
    label = row['label']

    if row.isna().any():
        rows.append({
            'label'               : label,
            'local_density_apical': np.nan,
            'local_density_basal' : np.nan,
            'fraccion_area_apical': np.nan,
            'fraccion_area_basal' : np.nan,
        })
        continue

    cz, cy, cx = centroids[label]
    cy, cx = round(cy), round(cx)

    # XY limits: disk cropped to the image boundary
    y0 = max(cy - radio_xy, 0);  y1 = min(cy + radio_xy + 1, labels.shape[1])
    x0 = max(cx - radio_xy, 0);  x1 = min(cx + radio_xy + 1, labels.shape[2])

    # Circular mask in YX
    yy, xx = np.ogrid[y0:y1, x0:x1]
    circle = ((yy - cy)**2 + (xx - cx)**2) <= radio_xy**2 #circular mask of pixels within the radio_xy around the centroid in the XY projection

    area_disco          = int(circle.sum()) * res_xy**2 # area of the disk in µm², which can be smaller than the area of a complete disk if it is near the edge of the image
    area_disco_completo = np.pi * radio_xy**2 * res_xy**2 # area of the complete disk in µm²

    dens_apical, frac_apical = density_2d(label, apical_map, y0, y1, x0, x1, circle, area_disco, area_disco_completo)
    dens_basal,  frac_basal  = density_2d(label, basal_map, y0, y1, x0, x1, circle, area_disco, area_disco_completo)

    rows.append({
        'label'               : label,
        'local_density_apical': dens_apical,
        'local_density_basal' : dens_basal,
        'fraccion_area_apical': frac_apical,
        'fraccion_area_basal' : frac_basal,
    })

df_density = pd.DataFrame(rows)
df_props = df_props.drop(columns=['local_density_apical', 'local_density_basal',
                                  'fraccion_area_apical', 'fraccion_area_basal'], errors='ignore')
df_props = df_props.merge(df_density, on='label')
print(df_props[['label', 'local_density_apical', 'local_density_basal',
               'fraccion_area_apical', 'fraccion_area_basal']].head(10))

## Feature representation

In [ ]:
print(df_props.describe().round(3))

In [ ]:
# Distribution of cellular features, identification of outliers, etc.
fig, axes = plt.subplots(6, 4, figsize=(12, 8))
axes = axes.ravel()

df_props['centroid_cell_z_norm_um'].hist(bins=100, ax=axes[0])
axes[0].set_title('Cell centroid Z relative to the epithelial base')

df_props['height_um'].hist(bins=100, ax=axes[1])
axes[1].set_title('Cell height')

df_props['area_apical_um2'].hist(bins=100, ax=axes[2])
axes[2].set_title('Apical area (µm²)')

df_props['area_basal_um2'].hist(bins=100, ax=axes[3])
axes[3].set_title('Basal area (µm²)')

df_props['logratio_ap_bas'].hist(bins=100, ax=axes[4])
axes[4].set_title('Apical-to-basal log ratio')

df_props['shape_index_apical'].hist(bins=100, ax=axes[5])
axes[5].set_title('Apical shape index')

df_props['shape_index_basal'].hist(bins=100, ax=axes[6])
axes[6].set_title('Basal shape index')

df_props['volume_um3'].hist(bins=100, ax=axes[7])
axes[7].set_title('Cell volume (µm³)')

df_props['surface_um2'].hist(bins=100, ax=axes[8])
axes[8].set_title('Cell surface area (µm²)')

df_props['sphericity'].hist(bins=100, ax=axes[9])
axes[9].set_title('Sphericity')

df_props['local_density_apical'].hist(bins=100, ax=axes[10])
axes[10].set_title('Apical local density\n(neighbors within a 75 px radius)')

df_props['local_density_basal'].hist(bins=100, ax=axes[11])
axes[11].set_title('Basal local density\n(neighbors within a 75 px radius)')

df_props['elongation'].hist(bins=100, ax=axes[12])
axes[12].set_title('Elongation')

df_props['angle_z'].hist(bins=100, ax=axes[13])
axes[13].set_title('Vertical orientation')

df_props['curvature_apical_tisular_mean'].hist(bins=100, ax=axes[14])
axes[14].set_title('Mean tissue-level apical curvature')

df_props['curvature_apical_celular_mean'].hist(bins=100, ax=axes[15])
axes[15].set_title('Mean cell-level apical curvature')

df_props['curvature_basal_tisular_inv_mean'].hist(bins=100, ax=axes[16])
axes[16].set_title('Mean tissue-level basal curvature')

df_props['curvature_basal_celular_inv_mean'].hist(bins=100, ax=axes[17])
axes[17].set_title('Mean cell-level basal curvature')

df_props['apical_neighbors_n'].hist(bins=100, ax=axes[20])
axes[20].set_title('Number of apical neighbors')

df_props['basal_neighbors_n'].hist(bins=100, ax=axes[21])
axes[21].set_title('Number of basal neighbors')

df_props['dif_neighbors'].hist(bins=100, ax=axes[22])
axes[22].set_title('Difference in neighbor number\nbetween apical and basal surfaces')

df_props['3D_neighbors_n'].hist(bins=100, ax=axes[23])
axes[23].set_title('Number of 3D neighbors')

plt.tight_layout()
plt.show()

print(df_props.drop(columns=['label']).describe())

## Filt columns, rows and export data

In [ ]:
df_props.to_csv(f"{date}_{file_name_root}_properties_full.csv", index=False)

In [ ]:
df_props.columns

In [ ]:
# Outlier and border filtering

# Select only the labels present in labels_noborder
df_props_final = df_props.copy()
df_props_final = df_props_final[df_props_final['border'] == False].copy()
print(f"Cells removed for being located at the border: {len(df_props) - len(df_props_final)}")

# Remove cells with extremely low cell volume
n_antes = len(df_props_final)
umbral_volumen = 50  # µm³
df_props_final = df_props_final[df_props_final['volume_um3'] > umbral_volumen]
print(f"Cells removed due to volume < {umbral_volumen} µm³: {n_antes - len(df_props_final)}")

# Remove extreme outliers based on differences in neighbor number (dif_neighbors)
n_antes = len(df_props_final)
umbral_dif_vecinos = 4
df_props_final = df_props_final[df_props_final['dif_neighbors'] < umbral_dif_vecinos]
print(f"Cells removed due to dif_neighbors >= {umbral_dif_vecinos}: {n_antes - len(df_props_final)}")

# Remove cells with an extremely high number of 3D, apical, or basal neighbors, which are likely segmentation artefacts
n_antes = len(df_props_final)
umbral_vecinos_3d = 11
df_props_final = df_props_final[df_props_final['3D_neighbors_n'] < umbral_vecinos_3d]
print(f"Cells removed due to 3D neighbors >= {umbral_vecinos_3d}: {n_antes - len(df_props_final)}")

n_antes = len(df_props_final)
umbral_vecinos_apicales = 11
df_props_final = df_props_final[df_props_final['apical_neighbors_n'] < umbral_vecinos_apicales]
print(f"Cells removed due to apical neighbors >= {umbral_vecinos_apicales}: {n_antes - len(df_props_final)}")

n_antes = len(df_props_final)
umbral_vecinos_basales = 11
df_props_final = df_props_final[df_props_final['basal_neighbors_n'] < umbral_vecinos_basales]
print(f"Cells removed due to basal neighbors >= {umbral_vecinos_basales}: {n_antes - len(df_props_final)}")

# Remove cells with an extremely low number of 3D, apical, or basal neighbors, which are likely segmentation artefacts
n_antes = len(df_props_final)
umbral_vecinos_3d = 2
df_props_final = df_props_final[df_props_final['3D_neighbors_n'] > umbral_vecinos_3d]
print(f"Cells removed due to 3D neighbors <= {umbral_vecinos_3d}: {n_antes - len(df_props_final)}")

n_antes = len(df_props_final)
umbral_vecinos_apicales = 2
df_props_final = df_props_final[df_props_final['apical_neighbors_n'] > umbral_vecinos_apicales]
print(f"Cells removed due to apical neighbors <= {umbral_vecinos_apicales}: {n_antes - len(df_props_final)}")

n_antes = len(df_props_final)
umbral_vecinos_basales = 2
df_props_final = df_props_final[df_props_final['basal_neighbors_n'] > umbral_vecinos_basales]
print(f"Cells removed due to basal neighbors <= {umbral_vecinos_basales}: {n_antes - len(df_props_final)}")

# Remove cells with a value in 'Inconsistency' different from 0
n_antes = len(df_props_final)
df_props_final = df_props_final[df_props_final['inconsistency'] == 0]
print(f"Cells removed due to inconsistencies between 3D and apical/basal neighbors: {n_antes - len(df_props_final)}")

# Remove cells with an extremely high or low apical-to-basal ratio, which are likely segmentation artefacts
n_antes = len(df_props_final)
umbral_logratio_ap_bas_alto = np.log2(3.0)
df_props_final = df_props_final[df_props_final['logratio_ap_bas'] < umbral_logratio_ap_bas_alto]
umbral_logratio_ap_bas_bajo = np.log2(0.33)
df_props_final = df_props_final[df_props_final['logratio_ap_bas'] > umbral_logratio_ap_bas_bajo]
print(f"Cells removed due to an apical-to-basal log ratio > {umbral_logratio_ap_bas_alto} or < {umbral_logratio_ap_bas_bajo}: "
      f"{n_antes - len(df_props_final)}")

# Retain only rows without NaN values (here, cells with apical or basal areas equal to 0,
# undefined curvature values, or neighbors that could not be calculated will be removed)
n_antes = len(df_props_final)
df_props_final = df_props_final.dropna().copy()
print(f"Cells removed due to NaN values in any property: {n_antes - len(df_props_final)}")

# Replace sphericity values greater than 1 with NaN, as they are not physically possible
# and likely indicate segmentation or calculation errors
df_props_final.loc[df_props_final['sphericity'] >= 1, 'sphericity'] = np.nan
print(f"Cells with sphericity >= 1: {sum(df_props_final['sphericity'] >= 1)}")

# Remove auxiliary columns that do not provide relevant information 
df_props_final.drop(columns=['centroid_cell_z', 'border', 'fraccion_area_basal',
                             'fraccion_area_apical', 'inconsistency', 'inconsistent_neighbors'],
                    inplace=True)

print(f"Labels in df_props: {len(df_props)}")
print(f"Labels in df_props_final: {len(df_props_final)}")
print(f"Labels removed: {len(df_props) - len(df_props_final)}")

# Save the results as a CSV file for downstream analyses
df_props_final.to_csv(f"{date}_{file_name_root}_properties_full_clean.csv", index=False)

In [ ]:
df_props_final.columns

In [ ]:
# Distribution of cellular features, identification of outliers, etc.
fig, axes = plt.subplots(6, 4, figsize=(12, 8))
axes = axes.ravel()

df_props_final['centroid_cell_z_norm_um'].hist(bins=100, ax=axes[0])
axes[0].set_title('Cell centroid Z relative to epithelial base')

df_props_final['height_um'].hist(bins=100, ax=axes[1])
axes[1].set_title('Cell height')

df_props_final['area_apical_um2'].hist(bins=100, ax=axes[2])
axes[2].set_title('Apical area (µm²)')

df_props_final['area_basal_um2'].hist(bins=100, ax=axes[3])
axes[3].set_title('Basal area (µm²)')

df_props_final['logratio_ap_bas'].hist(bins=100, ax=axes[4])
axes[4].set_title('Apical-Basal Log2ratio')

df_props_final['shape_index_apical'].hist(bins=100, ax=axes[5])
axes[5].set_title('Apical shape index')

df_props_final['shape_index_basal'].hist(bins=100, ax=axes[6])
axes[6].set_title('Basal shape index')

df_props_final['volume_um3'].hist(bins=100, ax=axes[7])
axes[7].set_title('Cell volume (µm³)')

df_props_final['surface_um2'].hist(bins=100, ax=axes[8])
axes[8].set_title('Cell surface area (µm²)')

df_props_final['sphericity'].hist(bins=100, ax=axes[9])
axes[9].set_title('Sphericity')

df_props_final['local_density_apical'].hist(bins=100, ax=axes[10])
axes[10].set_title('Local apical density (neighbors within 75 px radius)')

df_props_final['local_density_basal'].hist(bins=100, ax=axes[11])
axes[11].set_title('Local basal density (neighbors within 75 px radius)')

df_props_final['elongation'].hist(bins=100, ax=axes[12])
axes[12].set_title('Elongation')

df_props_final['angle_z'].hist(bins=100, ax=axes[13])
axes[13].set_title('Vertical orientation')

df_props_final['curvature_apical_tisular_mean'].hist(bins=100, ax=axes[14])
axes[14].set_title('Mean apical tissue curvature')

df_props_final['curvature_apical_celular_mean'].hist(bins=100, ax=axes[15])
axes[15].set_title('Mean apical cellular curvature')

df_props_final['curvature_basal_tisular_inv_mean'].hist(bins=100, ax=axes[16])
axes[16].set_title('Mean basal tissue curvature')

df_props_final['curvature_basal_celular_inv_mean'].hist(bins=100, ax=axes[17])
axes[17].set_title('Mean basal cellular curvature')

df_props_final['apical_neighbors_n'].hist(bins=100, ax=axes[20])
axes[20].set_title('Number of apical neighbors')

df_props_final['basal_neighbors_n'].hist(bins=100, ax=axes[21])
axes[21].set_title('Number of basal neighbors')

df_props_final['dif_neighbors'].hist(bins=100, ax=axes[22])
axes[22].set_title('Difference in neighbors between apical and basal')

df_props_final['3D_neighbors_n'].hist(bins=100, ax=axes[23])
axes[23].set_title('Number of 3D neighbors')

plt.tight_layout()
plt.show()

print(df_props_final.drop(columns=['label']).describe())

In [ ]:
#Keep only the columns of interest for downstream analyses, removing auxiliary columns that are not relevant
df_props_final_q = df_props_final.drop(columns=['apical_neighbors_labels', 'basal_neighbors_labels', '3D_neighbors_labels'], errors='ignore')

#Final check of the columns in the final dataframe
print(df_props_final_q.columns)

In [ ]:
df_props_final_q.to_csv(f"{date}_{file_name_root}_properties_quantitative.csv", index=False)